# 04e | Lightweight financial-model handoff and SFT on Kaggle

Use the private lightweight package exported from Colab, with trained `domain_v1` weights and the original tokenizer.
The complete 1.54 GiB transfer package is unnecessary. The first handoff imports only model weights and creates a new optimizer;
it is not exact cross-platform optimizer recovery. Model and label packages are private; use Private datasets and notebooks.

Attach the private lightweight dataset on Kaggle and select GPU T4 x2 with Internet on.
Only the first GPU is used. Run cells in order and enter the package SHA256 in the configuration cell.
The package retains source-verification reports from the Colab export, but Kaggle cannot reread original manifests that were not transferred.
Results concern exploratory headline literal-entity stance, not security prediction or X validation.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/kaggle/working/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/kaggle/working/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/kaggle/working")
print(f"Free disk space in /kaggle/working:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
from finllm.kaggle_lite import install_lite_archive, install_lite_directory, prepare_lite_sft
from finllm.training import run_training
from finllm.checkpoint import load_checkpoint, sha256_file
from finllm.runtime import restore_seed_checkpoint
from finllm.evaluation import run_evaluation
import torch

# Copy the manifest SHA256 from the lightweight export report; use the same inputs and code revision for recovery.
LITE_MANIFEST_SHA256 = "PASTE_LITE_MANIFEST_SHA256".strip().lower()
# Enter the ZIP SHA256 only if Kaggle did not extract the ZIP automatically.
LITE_ZIP_SHA256 = "".strip().lower()
INPUT = Path("/kaggle/input")
LITE = Path("/kaggle/working/130u_lite_input")
RUN = Path("/kaggle/working/130u_lite_sft_v1")
manifests = sorted(INPUT.rglob("lite_manifest.json"))
archives = sorted(INPUT.rglob("130u_kaggle_lite.zip"))
if len(manifests) == 1:
    installed = install_lite_directory(manifests[0].parent, LITE,
                                      expected_manifest_sha256=LITE_MANIFEST_SHA256)
elif not manifests and len(archives) == 1:
    installed = install_lite_archive(archives[0], LITE, expected_zip_sha256=LITE_ZIP_SHA256)
else:
    raise RuntimeError("Use Add Input to attach exactly one Private lightweight dataset.")
print("Lightweight input verification:", json.dumps(installed, indent=2, ensure_ascii=False))

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Select T4 GPU and rerun from the first cell.")
print("GPU count:", torch.cuda.device_count(), "Using:", torch.cuda.get_device_name(0))
pilot = prepare_lite_sft(PROJECT, LITE, RUN,
                         expected_manifest_sha256=installed["manifest_sha256"], epochs=2)
config = pilot["config"]
print("Task partitions:", json.dumps(pilot["counts"], indent=2, ensure_ascii=False))
latest = restore_seed_checkpoint(RUN, Path(config["backup_dir"]))
if latest:
    previous = load_checkpoint(latest, allow_previous=False)
    if previous.get("config", {}).get("lite_binding") != config.get("lite_binding"):
        raise ValueError("The existing checkpoint and lightweight input identities differ; preserve the original experiment and stop.")
    config.pop("init_checkpoint", None)
    config["resume"] = str(latest)
    print("Existing checkpoint detected; restore strictly with the same runtime environment.")
result = run_training(config)
print("SFT training results:", json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
for name, checkpoint in (("domain_before", LITE / "weights/domain_weights.pt"),
                         ("sft_best", RUN / "best.pt")):
    metrics = run_evaluation(checkpoint, config["tokenizer"], config["validation_data"],
                             RUN / (name + "_development.json"), target_mode="literal_entity")
    print(name, "Development results (exploratory, not final test):", json.dumps(metrics, indent=2, ensure_ascii=False))
print("Model and training state:", RUN)
print("Tokenizer and transfer inputs:", LITE)
print("Recovery checkpoint SHA256:", sha256_file(RUN / "latest.pt"))
print("Recovery integrity marker SHA256:", sha256_file(RUN / "latest.pt.json"))
print("Save a Private notebook version with outputs, or download latest.pt and latest.pt.json;")
print("also preserve the Private lightweight dataset and code revision from the first cell. Working-directory persistence is not guaranteed.")

After disconnection, rerun from the first cell if files remain in the same session. Cross-session recovery requires preserving and restoring the original working directory,
pinning the first-cell `REF` to the original full 40-character commit, with matching device, PyTorch/CUDA, data and paths.
Original Drive weights remain unchanged. Development metrics are not final evaluation; original publisher rights require separate verification.